Data Loading and Preprocessing

In [2]:
import pandas as pd
import os
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec
import pinecone
import time
from sentence_transformers import SentenceTransformer

d:\projects\VectorDB Semantic Search\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(find_dotenv(), override=True)

True

In [4]:
files = pd.read_csv(r"D:\projects\VectorDB Semantic Search\course-descriptions.csv", encoding = "ANSI")

In [5]:
# creating a single data string
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
    the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [6]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis = 1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [7]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [8]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [9]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")
else:
    print(f"{index_name} not in index list.")

my-index successfully deleted.


In [10]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-fh1kny3.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [11]:
index = pc.Index(index_name)

Embedding the Data

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1397.07it/s]


In [13]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [14]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


Semantic Search

In [15]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [16]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_values = True
)

In [17]:
query_results

{'matches': [{'id': 'Machine Learning in Excel',
              'score': 0.354954749,
              'values': [-0.0183002334,
                         -0.0279485621,
                         -0.0253203679,
                         -0.0126938438,
                         -0.0240366589,
                         -0.0219841022,
                         -0.0511236712,
                         -0.0535800271,
                         0.00997648761,
                         0.0282286741,
                         -0.0408324525,
                         -0.036268644,
                         0.0683277473,
                         -0.0348471254,
                         -0.00728511391,
                         0.0366663,
                         -0.00331021589,
                         -0.00411817292,
                         -4.75617453e-05,
                         -0.0627969056,
                         0.08469598,
                         0.0300104693,
                         -0.0528304949,
 

In [18]:
score_threshold = 0.3
for match in query_results["matches"]:
    if match['score'] >= score_threshold:
        print(f"Matched item ID: {match['id']}, score: {match['score']}")

Matched item ID: Machine Learning in Excel, score: 0.354954749
Matched item ID: Machine Learning with K-Nearest Neighbors, score: 0.314137965
